In [5]:
import pandas as pd
import re
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
import os

# === CONFIG ===
INPUT_FILES = [
    "boqcomparativechart (2).xlsx"
]
OUTPUT_FILE = "2024_UDD_88969_1.xlsx"

def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

def extract_class(text):
    text = str(text).lower()
    if 'k-9' in text or 'k9' in text:
        return 'DI-K9'
    elif 'k-7' in text:
        return 'DI-K7'
    elif 'm.s' in text or 'ms' in text:
        return 'MS'
    elif 'hdpe' in text:
        return 'HDPE'
    elif 'cpvc' in text:
        return 'CPVC'
    elif 'pvc' in text:
        return 'PVC'
    return None

def extract_dia(text):
    match = re.search(r'(\d+)\s*mm', str(text).lower())
    return int(match.group(1)) if match else None

unit_map = {
    "rmt": "mtr",
    "rm": "mtr",
    "meter": "mtr",
    "mtr": "mtr"
}

results = []

for input_file in INPUT_FILES:
    excel = pd.ExcelFile(input_file)
    for sheet in excel.sheet_names:
        try:
            # USE header=5 as per the actual file structure
            df = pd.read_excel(input_file, sheet_name=sheet, header=5)
        except Exception as e:
            print(f"⚠️ Skipping {input_file} - {sheet}: {e}")
            continue

        # Skip sheets with missing important columns
        required_columns = ["Description of Work / Item(s)", "Units", "Estimated Rate", "No.of Qty"]
        if not all(col in df.columns for col in required_columns):
            continue

        # Standardize column names
        df = df.rename(columns={
            "Description of Work / Item(s)": "Item Description",
            "Units": "Units",
            "Estimated Rate": "Estimate Rate",
            "No.of Qty": "Quantity"
        })

        df["Item Description"] = df["Item Description"].astype(str)
        df["class"] = df["Item Description"].apply(extract_class)
        df["DIA"] = df["Item Description"].apply(extract_dia)
        df["Estimate Rate"] = pd.to_numeric(df["Estimate Rate"], errors='coerce')
        df["Quantity"] = pd.to_numeric(df["Quantity"], errors='coerce')

        df["Units"] = df["Units"].astype(str).str.lower().str.replace('.', '', regex=False).str.replace(' ', '')

        df["Units"] = df["Units"].map(unit_map).fillna(df["Units"])

        df_pipe_mtr = df[
            df["Item Description"].str.contains("pipe|di|k-9|k9|k-7|m.s|ms|hdpe|pvc|cpvc", case=False, na=False) &
            (df["Units"] == "mtr")
        ].copy()

        for _, row in df_pipe_mtr.iterrows():
            results.append({
                "File": os.path.basename(input_file),
                "Sheet": sheet,
                "BOQ_NO": "Single",
                "Item Description": clean_illegal_chars(row["Item Description"]),
                "DI-K9": "Yes" if row["class"] == "DI-K9" else "",
                "DI-K7": "Yes" if row["class"] == "DI-K7" else "",
                "MS": "Yes" if row["class"] == "MS" else "",
                "HDPE": "Yes" if row["class"] == "HDPE" else "",
                "CPVC": "Yes" if row["class"] == "CPVC" else "",
                "PVC": "Yes" if row["class"] == "PVC" else "",
                "DIA": row["DIA"],
                "Estimate Rate": row["Estimate Rate"],
                "Units": row["Units"],
                "Quantity": row["Quantity"]
            })

df_summary = pd.DataFrame(results)
df_summary.insert(0, "SL No", range(1, len(df_summary) + 1))
df_summary.to_excel(OUTPUT_FILE, index=False)

print(f"✅ Filtered items from all files/sheets saved to: {OUTPUT_FILE}")


✅ Filtered items from all files/sheets saved to: 2024_UDD_88969_1.xlsx
